In [ ]:
import pytesseract
from pdf2image import convert_from_path
from jiwer import cer, wer
import pandas as pd
from pathlib import Path

TESSERACT_CONFIG = r'--oem 3 --psm 6'

def normalize(text):
    return " ".join(text.lower().split())

def ocr_pdf(pdf_path, dpi=200):
    pages = convert_from_path(pdf_path, dpi=dpi)
    extracted_text = ""
    for page in pages:
        text = pytesseract.image_to_string(page, config=TESSERACT_CONFIG)
        extracted_text += text + "\n"
    return extracted_text

In [8]:
dataset_path = Path("dataset-png")
results = []

for pdf_path in dataset_path.glob("*.pdf"):
    gt_path = pdf_path.with_suffix(".txt")
    if not gt_path.exists():
        continue

    try:
        ground_truth = normalize(gt_path.read_text(encoding="utf-8"))
        ocr_text = normalize(ocr_pdf(pdf_path))
    except Exception as e:
        print(f"Error on {pdf_path.name}: {e}")
        continue

    results.append({
        "file": pdf_path.name,
        "CER": cer(ground_truth, ocr_text),
        "WER": wer(ground_truth, ocr_text)
    })

df = pd.DataFrame(results)

In [9]:
print("Average CER:", df["CER"].mean())
print("Average WER:", df["WER"].mean())
print(df.sort_values("WER", ascending=False))

Average CER: 0.04665670563890406
Average WER: 0.06504165719621838
                             file       CER       WER
26       billing_statement_27.pdf  0.047897  0.077519
19        billing_statement_9.pdf  0.049682  0.076271
24                          0.pdf  0.026743  0.075472
25       billing_statement_26.pdf  0.049939  0.072000
2        billing_statement_14.pdf  0.048220  0.068702
29       billing_statement_18.pdf  0.048611  0.068702
15        billing_statement_4.pdf  0.046719  0.068182
28       billing_statement_19.pdf  0.047458  0.068182
22       billing_statement_23.pdf  0.048276  0.068182
0        billing_statement_29.pdf  0.047191  0.067669
4        billing_statement_16.pdf  0.048000  0.067669
23       billing_statement_22.pdf  0.047458  0.067669
1        billing_statement_15.pdf  0.046563  0.067669
8        billing_statement_10.pdf  0.048000  0.067164
11        billing_statement_2.pdf  0.047727  0.067164
31       billing_statement_24.pdf  0.045356  0.066176
5        billing